# Load Model

In [1]:
import re
import os
import random
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
# from huggingface_hub import login
# from google.colab import userdata

seed = 42

# token = userdata.get('HF_TOKEN')
# if not token:
#     raise EnvironmentError("Set HF_TOKEN environment variable")

MODEL_ID = "CohereLabs/tiny-aya-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Cohere2ForCausalLM(
  (model): Cohere2Model(
    (embed_tokens): Embedding(262144, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-35): 36 x Cohere2DecoderLayer(
        (self_attn): Cohere2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Cohere2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Cohere2LayerNorm()
      )
    )
    (norm): Cohere2LayerNorm()
    (rotary_emb): Cohere2RotaryEmbedding()
  )
  (lm_head): Linear(in_featur

Shared Helpers

In [2]:
def generate_text(prompt: str, max_new_tokens: int = 80) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()

# Load Datasets

MGSM

In [3]:
mgsm_zh = pd.read_csv("/content/mgsm_zh.tsv", sep="\t", header=None)
mgsm_zh = mgsm_zh.rename(columns={0: "Question", 1: "Answer"})

mgsm_es = pd.read_csv("/content/mgsm_es.tsv", sep="\t", header=None)
mgsm_es = mgsm_es.rename(columns={0: "Question", 1: "Answer"})

mgsm_ur_ds = load_dataset("large-traversaal/mgsm_urdu_final")
mgsm_ur = mgsm_ur_ds["test"].to_pandas()
mgsm_ur = mgsm_ur.drop(columns=["question", "answer", "urdu_answer", "equation_solution"])
mgsm_ur = mgsm_ur.rename(columns={"urdu_question": "Question", "answer_number": "Answer"})

Repo card metadata block was not found. Setting CardData to empty.


XNLI

In [4]:
xnli_zh = load_dataset("facebook/xnli", "zh")["test"].to_pandas()
xnli_es = load_dataset("facebook/xnli", "es")["test"].to_pandas()
xnli_ur = load_dataset("facebook/xnli", "ur")["test"].to_pandas()

CSQA

In [5]:
csqa_es = load_dataset("INK-USC/xcsr", "X-CSQA-es")["validation"].to_pandas()
csqa_zh = load_dataset("INK-USC/xcsr", "X-CSQA-zh")["validation"].to_pandas()
csqa_ur = load_dataset("INK-USC/xcsr", "X-CSQA-ur")["validation"].to_pandas()

def prepare_csqa(df):
    df = df[["question", "answerKey"]].copy()
    df["stem"] = df["question"].apply(lambda x: x["stem"])
    df["choice_texts"] = df["question"].apply(lambda x: list(x["choices"]["text"]))

    df["A"] = df["choice_texts"].apply(lambda x: x[0])
    df["B"] = df["choice_texts"].apply(lambda x: x[1])
    df["C"] = df["choice_texts"].apply(lambda x: x[2])
    df["D"] = df["choice_texts"].apply(lambda x: x[3])
    df["E"] = df["choice_texts"].apply(lambda x: x[4])

    return df[["stem", "A", "B", "C", "D", "E", "answerKey"]]

csqa_es = prepare_csqa(csqa_es)
csqa_zh = prepare_csqa(csqa_zh)
csqa_ur = prepare_csqa(csqa_ur)

MGSM Functions

In [6]:
def build_mgsm_prompt(question: str, lang) -> str:
    templates = {
        "en": f"""Solve the following math word problem.
Give the final answer as a number only.

Question: {question}
Answer:""",
        "es": f"""Resuelve el siguiente problema matemático.
Da la respuesta final solo como un número.

Pregunta: {question}
Respuesta:""",
        "zh": f"""请解答下面的数学应用题。
最终答案只输出数字。

题目: {question}
答案:""",
        "ur": f"""مندرجہ ذیل ریاضی کے سوال کو حل کریں۔
صرف عددی شکل میں آخری جواب دیں۔

سوال: {question}
جواب:"""
    }
    return templates[lang]

def extract_number(text: str):
    matches = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return matches[-1] if matches else None

def normalize_number_string(x):
    x = str(x).strip().replace(",", "")
    matches = re.findall(r"-?\d+(?:\.\d+)?", x)
    return matches[-1] if matches else None

def evaluate_mgsm(df: pd.DataFrame, lang, n_samples=None):
    total, correct = 0, 0
    rows = []

    eval_df = df if n_samples is None else df.head(n_samples)

    for _, row in eval_df.iterrows():
        prompt = build_mgsm_prompt(row["Question"], lang)
        output = generate_text(prompt, max_new_tokens=80)
        pred = extract_number(output)
        gold = normalize_number_string(row["Answer"])

        is_correct = pred == gold
        correct += int(is_correct)
        total += 1

        rows.append({
            "question": row["Question"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / total if total else 0.0, pd.DataFrame(rows)

XNLI Functions

In [7]:
LABEL_MAP = {
    0: "entailment",
    1: "neutral",
    2: "contradiction"
}

def build_xnli_prompt(row, lang):
    templates = {
        "en": f"""Read the premise and hypothesis below.
Decide whether the hypothesis is entailed by the premise, contradicted by the premise, or neither.
Reply with only one word: entailment, contradiction, or neutral.

Premise: {row['premise']}
Hypothesis: {row['hypothesis']}

Answer:""",

        "es": f"""Lee la premisa y la hipótesis a continuación.
Decide si la hipótesis se sigue de la premisa, la contradice, o no es ninguna de las dos.
Responde con solo una palabra: entailment, contradiction, o neutral.

Premisa: {row['premise']}
Hipótesis: {row['hypothesis']}

Respuesta:""",

        "zh": f"""请阅读下面的前提和假设。
判断假设是否被前提蕴含、与前提矛盾，或两者都不是。
只回答一个词：entailment、contradiction 或 neutral。

前提: {row['premise']}
假设: {row['hypothesis']}

答案:""",

        "ur": f"""نیچے دی گئی premise اور hypothesis کو پڑھیں۔
فیصلہ کریں کہ hypothesis، premise سے لازم آتی ہے، اس کی تردید کرتی ہے، یا دونوں میں سے کوئی نہیں۔
صرف ایک لفظ میں جواب دیں: entailment، contradiction، یا neutral۔

Premise: {row['premise']}
Hypothesis: {row['hypothesis']}

جواب:"""
    }
    return templates[lang]


def extract_xnli_label(text: str):
    text = text.strip().lower()
    for label in ["entailment", "contradiction", "neutral"]:
        if re.search(rf"\b{label}\b", text):
            return label
    return None

def normalize_xnli_gold(label):
    if isinstance(label, str):
        return label.strip().lower()
    return LABEL_MAP.get(int(label), None)

def evaluate_xnli(df: pd.DataFrame, lang, n_samples=None):
    total, correct = 0, 0
    rows = []

    eval_df = df if n_samples is None else df.head(n_samples)

    for _, row in eval_df.iterrows():
        prompt = build_xnli_prompt(row, lang)
        output = generate_text(prompt, max_new_tokens=10)
        pred = extract_xnli_label(output)
        gold = normalize_xnli_gold(row["label"])

        is_correct = pred == gold
        correct += int(is_correct)
        total += 1

        rows.append({
            "premise": row["premise"],
            "hypothesis": row["hypothesis"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / total if total else 0.0, pd.DataFrame(rows)

CSQA Functions

In [8]:
def build_csqa_prompt(row, lang):
    templates = {
        "en": f"""Choose the best answer to the following commonsense question.
Reply with only one letter: A, B, C, D, or E.

Question: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

Answer:""",

        "es": f"""Elige la mejor respuesta para la siguiente pregunta de sentido común.
Responde solo con una letra: A, B, C, D o E.

Pregunta: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

Respuesta:""",

        "zh": f"""请选择下面这个常识问题的最佳答案。
只回答一个字母：A、B、C、D 或 E。

问题: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

答案:""",

        "ur": f"""نیچے دیے گئے عمومی فہم کے سوال کا بہترین جواب منتخب کریں۔
صرف ایک حرف میں جواب دیں: A، B، C، D، یا E۔

سوال: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

جواب:"""
    }
    return templates[lang]

def extract_choice(text: str):
    text = text.strip().upper()

    match = re.search(r"\b([ABCDE])\b", text)
    if match:
        return match.group(1)

    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCDE])", text)
    if match:
        return match.group(1)

    return None

def evaluate_csqa(df: pd.DataFrame, lang, n_samples=None):
    total, correct = 0, 0
    rows = []

    eval_df = df if n_samples is None else df.head(n_samples)

    for _, row in eval_df.iterrows():
        prompt = build_csqa_prompt(row, lang)
        output = generate_text(prompt, max_new_tokens=10)
        pred = extract_choice(output)
        gold = str(row["answerKey"]).strip().upper()

        is_correct = pred == gold
        correct += int(is_correct)
        total += 1

        rows.append({
            "stem": row["stem"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / total if total else 0.0, pd.DataFrame(rows)

English Evals

In [9]:
def main():
    summary = {}

    # MGSM
    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(mgsm_zh, lang = 'en', n_samples=5)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(mgsm_es, lang = 'en', n_samples=5)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(mgsm_ur, lang = 'en', n_samples=5)

    # XNLI
    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(xnli_zh, lang = 'en', n_samples=5)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(xnli_es, lang = 'en', n_samples=5)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(xnli_ur, lang = 'en', n_samples=5)

    # CSQA
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(csqa_es, lang = 'en', n_samples=5)
    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(csqa_zh, lang = 'en', n_samples=5)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(csqa_ur, lang = 'en', n_samples=5)

    print("Summary:")
    for k, v in summary.items():
        print(f"{k}: {v:.4f}")

    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh,
        "mgsm_es": res_mgsm_es,
        "mgsm_ur": res_mgsm_ur,
        "xnli_zh": res_xnli_zh,
        "xnli_es": res_xnli_es,
        "xnli_ur": res_xnli_ur,
        "csqa_es": res_csqa_es,
        "csqa_zh": res_csqa_zh,
        "csqa_ur": res_csqa_ur,
    }

if __name__ == "__main__":
    results = main()

Summary:
mgsm_zh_acc: 0.0000
mgsm_es_acc: 0.2000
mgsm_ur_acc: 0.0000
xnli_zh_acc: 0.4000
xnli_es_acc: 0.4000
xnli_ur_acc: 0.2000
csqa_es_acc: 0.8000
csqa_zh_acc: 0.6000
csqa_ur_acc: 0.0000


Language Specific Evals

In [32]:
def main():
    summary = {}

    # MGSM
    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(mgsm_zh, lang = 'zh', n_samples=5)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(mgsm_es, lang = 'es', n_samples=5)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(mgsm_ur, lang = 'ur', n_samples=5)

    # XNLI
    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(xnli_zh, lang = 'zh', n_samples=5)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(xnli_es, lang = 'es', n_samples=5)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(xnli_ur, lang = 'ur', n_samples=5)

    # CSQA
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(csqa_es, lang = 'zh', n_samples=5)
    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(csqa_zh, lang = 'es', n_samples=5)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(csqa_ur, lang = 'ur', n_samples=5)

    print("Summary:")
    for k, v in summary.items():
        print(f"{k}: {v:.4f}")

    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh,
        "mgsm_es": res_mgsm_es,
        "mgsm_ur": res_mgsm_ur,
        "xnli_zh": res_xnli_zh,
        "xnli_es": res_xnli_es,
        "xnli_ur": res_xnli_ur,
        "csqa_es": res_csqa_es,
        "csqa_zh": res_csqa_zh,
        "csqa_ur": res_csqa_ur,
    }

if __name__ == "__main__":
    results = main()

Summary:
mgsm_zh_acc: 0.0000
mgsm_es_acc: 0.2000
mgsm_ur_acc: 0.0000
xnli_zh_acc: 0.0000
xnli_es_acc: 0.0000
xnli_ur_acc: 0.2000
csqa_es_acc: 0.8000
csqa_zh_acc: 0.6000
csqa_ur_acc: 0.0000
